# Kriging with the Vecchia log-likelihood — objective="VLL(m)" (Python)

This notebook demonstrates the **Vecchia approximated log-likelihood** as a fit
objective. Vecchia (1988): log p(y) is approximated by the product of the
conditionals of each observation given its $m$ nearest previously-ordered
neighbors (greedy maxmin ordering, Guinness 2018):

$$\log L \approx \sum_i \log p(y_i \mid y_{N(i)}), \qquad |N(i)| \le m$$

Each evaluation costs $O(n\,m^3)$ instead of $O(n^3)$, is a valid Gaussian
density (sparse inverse Cholesky), and is exact for $m = n-1$. The usage could
not be simpler — it is just an `objective` string:

- `objective="VLL"` (default $m=30$) or `objective="VLL(m)"`.

After the fit, ONE exact factorization is performed at $\theta^*$, so
`predict`/`simulate`/`update` behave exactly as after an `"LL"` fit.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 1500$)
4. Fit with `"VLL(30)"` vs exact `"LL"`: wall time and hyperparameters
5. Predictions: both models on a grid, RMSE
6. Effect of the number of neighbors $m$


## 0. Installation

Set `REPO_ROOT`, then optionally create a venv and install build requirements
(skip the build cells if pylibkriging is already installed).

In [ ]:
%%bash
REPO_ROOT=$(cd ../.. && pwd)
echo "REPO_ROOT=${REPO_ROOT}"

In [ ]:
%%bash
# Optional: skip if pylibkriging is already installed
set -e
REPO_ROOT=$(cd ../.. && pwd)
VENV_DIR=./venv

# Create venv if needed
if [ ! -d "${VENV_DIR}" ]; then
    python3 -m venv "${VENV_DIR}"
fi
source "${VENV_DIR}/bin/activate"

# Install build requirements
pip install -q \
    -r "${REPO_ROOT}/bindings/Python/pylibkriging/requirements.txt" \
    -r "${REPO_ROOT}/bindings/Python/pylibkriging/dev-requirements.txt"

pip install matplotlib

In [ ]:
%%bash
# Optional: compile libkriging from source if not already built
set -e
REPO_ROOT=$(cd ../.. && pwd)

    cd "${REPO_ROOT}"
    # Point repo-level venv to our local one so loadenv.sh picks it up
    if [ ! -e venv ]; then
        ln -s bindings/Python/venv venv
    fi
    source venv/bin/activate
    
if [ -d "${REPO_ROOT}/build/installed" ]; then
    echo "libkriging already built, skipping build step"
else
    # Force cmake to use the venv python
    EXTRA_CMAKE_OPTIONS="-DPYTHON_EXECUTABLE=$(which python3)" \
        ENABLE_PYTHON_BINDING=on BUILD_TEST=false \
        tools/linux-macos/build.sh
fi


In [ ]:
%%bash
REPO_ROOT=$(cd ../.. && pwd)
# Optional: skip if pylibkriging is already installed
pip install --no-build-isolation ${REPO_ROOT}/bindings/Python/pylibkriging/

## 1. Load pylibkriging

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylibkriging as lk

print("pylibkriging version:", lk.__version__)

## 2. Branin function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def branin(X):
    x1 = X[:, 0] * 15 - 5
    x2 = X[:, 1] * 15
    return (x2 - 5 / (4 * np.pi**2) * x1**2 + 5 / np.pi * x1 - 6)**2 \
           + 10 * (1 - 1 / (8 * np.pi)) * np.cos(x1) + 10

grid_x = np.linspace(0, 1, 50)
G1, G2 = np.meshgrid(grid_x, grid_x)
grid_pts = np.column_stack([G1.ravel(), G2.ravel()])
z_true = branin(grid_pts).reshape(50, 50)

plt.contourf(G1, G2, z_true, 20)
plt.colorbar(); plt.title("True Branin"); plt.show()

## 3. Design of experiments ($n = 1500$)

Large enough that each exact likelihood evaluation factorizes a
$1500\times1500$ matrix, while a Vecchia evaluation only factorizes $n$ small
$30\times30$ blocks.

In [ ]:
rng = np.random.default_rng(123)
n = 1500
X = rng.uniform(size=(n, 2))
y = branin(X)
print("design:", X.shape)

## 4. Fit: `VLL(30)` vs exact `LL`

In [ ]:
t0 = time.time()
k_vll = lk.Kriging(y, X, "matern5_2", objective="VLL(30)")
t_vll = time.time() - t0

t0 = time.time()
k_ll = lk.Kriging(y, X, "matern5_2", objective="LL")
t_ll = time.time() - t0

print(f"fit VLL(30) : {t_vll:6.1f} s   theta = {k_vll.theta().ravel()}")
print(f"fit LL      : {t_ll:6.1f} s   theta = {k_ll.theta().ravel()}")
print(f"speedup     : {t_ll / t_vll:.1f}x")

## 5. Predictions

Both models use the exact predictor at their own $\theta^*$.

In [ ]:
p_vll = k_vll.predict(grid_pts, True, False, False)
p_ll = k_ll.predict(grid_pts, True, False, False)
m_vll, m_ll = np.asarray(p_vll[0]).ravel(), np.asarray(p_ll[0]).ravel()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
c0 = ax[0].contourf(G1, G2, m_vll.reshape(50, 50), 20)
fig.colorbar(c0, ax=ax[0]); ax[0].set_title("mean — fit VLL(30)")
c1 = ax[1].contourf(G1, G2, m_ll.reshape(50, 50), 20)
fig.colorbar(c1, ax=ax[1]); ax[1].set_title("mean — fit LL")
plt.show()

print("RMSE vs true (VLL):", np.sqrt(np.mean((m_vll - z_true.ravel())**2)))
print("RMSE vs true (LL) :", np.sqrt(np.mean((m_ll - z_true.ravel())**2)))
print("max |VLL - LL|    :", np.abs(m_vll - m_ll).max())

## 6. Effect of the number of neighbors $m$

In [ ]:
for m in [5, 15, 30]:
    t0 = time.time()
    k = lk.Kriging(y, X, "matern5_2", objective=f"VLL({m})")
    p = k.predict(grid_pts, False, False, False)
    rmse = np.sqrt(np.mean((np.asarray(p[0]).ravel() - z_true.ravel())**2))
    print(f"VLL({m:2d}) : fit {time.time()-t0:6.1f} s   theta = {np.round(k.theta().ravel(), 4)}   RMSE = {rmse:.4f}")

## Notes

- The screening effect behind Vecchia weakens in high dimension: recommended
  for $d \le 5$ (complementary to `NestedKriging`, which is dimension-robust).
- The final exact factorization at $\theta^*$ still costs $O(n^3)$ once. For
  very large $n$, the C++ API offers `predictVecchia` (local $O(q\,m^3)$
  prediction) and a "light" mode (`set_vecchia_exact_commit(false)`) that
  skips it entirely — bindings for both are planned.
- References: Vecchia (1988); Guinness (2018), *Technometrics*; Katzfuss &
  Guinness (2021), *Statistical Science*.
